🚀 Great. You've completed:

Day 1 → EDA + popularity baseline
Day 2 → Weighted ratings
Day 3 → User collaborative filtering
Day 4 → Item collaborative filtering
Day 5 → SVD / Matrix factorization
Day 6 → Top-N recommendation engine
Day 7 → Content-based recommendation
Day 8 → Hybrid recommendation system

Now we begin shifting from:

Notebook experiments

toward:

Production-style ML system

🚀 Day 9 — Refactoring Into a Real Project Structure

Until now everything was mostly inside notebooks/scripts.

Real ML systems separate:

Data loading
Model training
Recommendation logic
API layer
Utilities

This makes the system:

✅ maintainable
✅ reusable
✅ deployable
✅ easier to debug


---

🧠 Goal Today

Convert recommendation logic into reusable Python modules.

Instead of:

# huge notebook

we move toward:

src/
    data_loader.py
    train_model.py
    recommender.py
    hybrid.py


---

Recommended project structure

recommendation_system/

│
├── data/
│
├── notebooks/
│
├── models/
│
├── api/
│
├── logs/
│
├── src/
│   ├── data_loader.py
│   ├── train_model.py
│   ├── collaborative.py
│   ├── content_based.py
│   ├── hybrid.py
│   └── utils.py
│
├── requirements.txt
│
└── main.py


---

Step 1 — Create data_loader.py

Move dataset loading logic.

src/data_loader.py

import pandas as pd


def load_data():

    ratings_cols = [
        'user_id',
        'movie_id',
        'rating',
        'timestamp'
    ]

    ratings = pd.read_csv(
        'data/ml-100k/u.data',
        sep='\t',
        names=ratings_cols
    )

    movie_cols = [
        'movie_id',
        'title',
        'release_date',
        'video_release',
        'IMDb_URL',
        'unknown',
        'Action',
        'Adventure',
        'Animation',
        'Children',
        'Comedy',
        'Crime',
        'Documentary',
        'Drama',
        'Fantasy',
        'Film_Noir',
        'Horror',
        'Musical',
        'Mystery',
        'Romance',
        'SciFi',
        'Thriller',
        'War',
        'Western'
    ]

    movies = pd.read_csv(
        'data/ml-100k/u.item',
        sep='|',
        encoding='latin-1',
        header=None,
        names=movie_cols
    )

    df = pd.merge(
        ratings,
        movies,
        on='movie_id'
    )

    return ratings,movies,df


---

Step 2 — Create collaborative.py

Move collaborative recommendation logic.

src/collaborative.py

from surprise import SVD
from surprise import Dataset
from surprise import Reader


def train_svd_model(ratings):

    reader = Reader(
        rating_scale=(1,5)
    )

    data = Dataset.load_from_df(
        ratings[
            [
                'user_id',
                'movie_id',
                'rating'
            ]
        ],
        reader
    )

    trainset = data.build_full_trainset()

    model = SVD()

    model.fit(trainset)

    return model


---

Step 3 — Create content_based.py

src/content_based.py

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity


def build_content_similarity(movies):

    genre_columns = movies.columns[5:]

    movies['genres'] = movies[
        genre_columns
    ].apply(
        lambda x:' '.join(
            x.index[x==1]
        ),
        axis=1
    )

    cv = CountVectorizer()

    movie_vectors = cv.fit_transform(
        movies['genres']
    )

    similarity = cosine_similarity(
        movie_vectors
    )

    return similarity


---

Step 4 — Create hybrid.py

src/hybrid.py

import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler


def hybrid_recommendation(
    user_id,
    liked_movies,
    ratings,
    movies,
    similarity,
    model
):

    watched_movies = ratings[
        ratings['user_id']==user_id
    ]['movie_id'].tolist()

    all_movies = ratings[
        'movie_id'
    ].unique()

    predictions=[]

    for movie in all_movies:

        if movie not in watched_movies:

            pred=model.predict(
                uid=user_id,
                iid=movie
            )

            predictions.append(
                (
                    movie,
                    pred.est
                )
            )

    collab_df = pd.DataFrame(
        predictions,
        columns=[
            'movie_id',
            'collab_score'
        ]
    )

    content_score=np.zeros(
        len(movies)
    )

    total_rating=0

    for movie in liked_movies:

        idx=movies[
            movies['title']==movie
        ].index[0]

        rating=ratings[
            (
                ratings['user_id']==user_id
            )
        ]['rating'].mean()

        content_score += (
            similarity[idx] * rating
        )

        total_rating += rating

    content_score = (
        content_score /
        total_rating
    )

    content_score = list(
        enumerate(
            content_score
        )
    )

    content_df = pd.DataFrame(
        content_score,
        columns=[
            'movie_index',
            'content_score'
        ]
    )

    content_df[
        'movie_id'
    ] = movies['movie_id']

    hybrid = pd.merge(
        collab_df,
        content_df,
        on='movie_id'
    )

    scaler = MinMaxScaler()

    hybrid[
        'collab_score'
    ] = scaler.fit_transform(
        hybrid[
            ['collab_score']
        ]
    )

    hybrid[
        'content_score'
    ] = scaler.fit_transform(
        hybrid[
            ['content_score']
        ]
    )

    hybrid[
        'final_score'
    ] = (
        0.7 * hybrid[
            'collab_score'
        ]
        +
        0.3 * hybrid[
            'content_score'
        ]
    )

    hybrid = hybrid.sort_values(
        'final_score',
        ascending=False
    )

    return hybrid.head(10)


---

Step 5 — Create main.py

main.py

from src.data_loader import load_data

from src.collaborative import train_svd_model

from src.content_based import build_content_similarity

from src.hybrid import hybrid_recommendation


ratings,movies,df = load_data()

model = train_svd_model(
    ratings
)

similarity = build_content_similarity(
    movies
)

user_id = 1

liked_movies = df[
    (
        df['user_id']==user_id
    )
].sort_values(
    'rating',
    ascending=False
)['title'].head(5).tolist()


recommendations = hybrid_recommendation(
    user_id,
    liked_movies,
    ratings,
    movies,
    similarity,
    model
)

print(
    recommendations
)


---

🧠 What changed today?

Before:

Everything mixed together

Now:

Reusable modules
↓
Cleaner architecture
↓
Production direction


---

Important software engineering concepts learned

1. Separation of concerns

Each file
=
One responsibility


---

2. Modular design

Train model separately
Use model separately


---

3. Reusability

You can now:

from src.hybrid import hybrid_recommendation

anywhere.


---

4. Maintainability

Future changes become easier.

Example:

Replace SVD
↓
Only edit collaborative.py

instead of entire notebook.


---

🎯 Homework

1. Create all files physically

Inside VS Code:

src/

Create:

data_loader.py
collaborative.py
content_based.py
hybrid.py


---

2. Run main.py

Check:

Do recommendations still work?


---

3. Think about this

Right now:

Training happens every run

Question:

Can we save trained models
instead of retraining?

That becomes:

🚀 Day 10 — Model Persistence (pickle / joblib)

